### The ```factive``` variable represents the fraction of active tracking wire hits used in a track reconstruction relative to the total expected hits.

In [7]:
import uproot
import joblib
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import pandas as pd


# 1. Define the file parameters
training_dataset_filename = "/Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training.root"
training_dataset_treename = "trkqualtree"

In [8]:
def add_useful_columns(batch):
    # Calculate magnitude of the momenta
    batch['trk_ent.mom'] = (batch['trk_ent.mom.fCoordinates.fX']**2 + batch['trk_ent.mom.fCoordinates.fY']**2 + batch['trk_ent.mom.fCoordinates.fZ']**2)**0.5
    batch['trk_ent_mc.mom'] = (batch['trk_ent_mc.mom.fCoordinates.fX']**2 + batch['trk_ent_mc.mom.fCoordinates.fY']**2 + batch['trk_ent_mc.mom.fCoordinates.fZ']**2)**0.5
    
    # For masks
    #batch['ent_fit_is_highmom'] = ak.flatten(ak.any( (batch['demfit.mom']>80) & (batch['demfit.sid']==0), axis=-1, keepdims=True))
    
    # For training features
    batch['trk.factive'] = batch['trk.nactive'] / batch['trk.nhits']
    batch['trk.fambig'] = batch['trk.nnullambig'] / batch['trk.nactive']
    batch['trk.fstraws'] = batch['trk.nmatactive'] / batch['trk.nactive']

In [9]:
def load_training_arrays_batched(filepath, treename, step_size="50 MB", start_code=173):
    tree = uproot.open(f"{filepath}:{treename}")
    print(f"{filepath.split('/')[-1]}: {tree.num_entries} entries")

    out = {k: [] for k in [
        "trk_ent_mom", "trk_ent_mc_mom", "nactive", "factive",
        "fambig", "fstraws", "t0err", "fitcon", "momerr"
    ]}

    needed = [
        # mask inputs
        "trk.status", "trk.goodfit", "trk_ent_pars.t0err", "trk_sim.startCode",

        # direct outputs / raw
        "trk.nactive", "trk.fitcon", "trk_ent.momerr",

        # required by add_useful_columns for derived feature columns
        "trk.nhits", "trk.nnullambig", "trk.nmatactive",

        # required by add_useful_columns for momentum magnitudes
        "trk_ent.mom.fCoordinates.fX",
        "trk_ent.mom.fCoordinates.fY",
        "trk_ent.mom.fCoordinates.fZ",
        "trk_ent_mc.mom.fCoordinates.fX",
        "trk_ent_mc.mom.fCoordinates.fY",
        "trk_ent_mc.mom.fCoordinates.fZ",
    ]

    for batch in tree.iterate(filter_name=needed, step_size=step_size, library="ak"):
        add_useful_columns(batch)

        t0 = batch["trk_ent_pars.t0err"]
        t0_np = ak.to_numpy(t0)  # 1D numeric after your selection style
        valid_t0 = np.isfinite(t0_np)   # handles NaN and inf
        mask = (
            (batch["trk.status"] > 0)
            & (batch["trk.goodfit"] == 1)
            & ~ak.is_none(t0)
            & valid_t0
            & (batch["trk_sim.startCode"] == start_code)
        )

        out["trk_ent_mom"].append(ak.to_numpy(batch["trk_ent.mom"][mask]))
        out["trk_ent_mc_mom"].append(ak.to_numpy(batch["trk_ent_mc.mom"][mask]))
        out["nactive"].append(ak.to_numpy(batch["trk.nactive"][mask]))
        out["factive"].append(ak.to_numpy(batch["trk.factive"][mask]))
        out["fambig"].append(ak.to_numpy(batch["trk.fambig"][mask]))
        out["fstraws"].append(ak.to_numpy(batch["trk.fstraws"][mask]))
        out["t0err"].append(ak.to_numpy(t0[mask]))
        out["fitcon"].append(ak.to_numpy(batch["trk.fitcon"][mask]))
        out["momerr"].append(ak.to_numpy(batch["trk_ent.momerr"][mask]))

    for k, chunks in out.items():
        out[k] = np.concatenate(chunks) if chunks else np.array([], dtype=np.float64)
    return out

# 4. Execute the loader tool to create data_dict globally in this notebook
data_dict = load_training_arrays_batched(training_dataset_filename, training_dataset_treename)

trkqual_tree_v2.0_training.root: 452686 entries


In [10]:
best_model = joblib.load("/Users/malikfarouh/Documents/ML workspace/ML Model/xgb_trkqual_best_model_v1.joblib")
booster = best_model.get_booster()

input_var_names = ["nactive", "factive", "t0err", "fambig", "fitcon", "momerr", "fstraws"]
booster.feature_names = input_var_names

print("Baseline model loaded successfully.")

Baseline model loaded successfully.


In [11]:
# 1. SETUP: Reassemble the full feature matrix from the data_dict in the original file
#input_var_names = ["nactive", "factive", "t0err", "fambig", "fitcon", "momerr", "fstraws"]
x_full = np.column_stack([data_dict[k] for k in input_var_names])

# 2. REGENERATE MASKS: Use the raw momentum arrays to find quality tracks 
# Since the data_dict contains the raw track info, we compute the masks here
mom_res = data_dict["trk_ent_mom"] - data_dict["trk_ent_mc_mom"]
high_qual = (mom_res > -0.25) & (mom_res < 0.25)
low_qual = mom_res > 0.7

# Assemble the full binary truth target array
y_full = np.zeros(len(x_full))
y_full[high_qual] = 1
y_full[low_qual] = 0

# 3. CHOOSE CUT POINT: We use the exact threshold value from the trained model which is 0.92

trkqual_cut = 0.92

# Generate fresh continuous baseline probabilities using the loaded best_model file
y_pred_prob = best_model.predict_proba(x_full)[:, 1]
bdt_pass = y_pred_prob >= trkqual_cut

# 4. STATISTICAL FUNCTIONS: Calculate fraction means and tracking uncertainties
x_full_df = pd.DataFrame(x_full, columns=input_var_names)
f_track_full = x_full_df['factive'].values

def f_and_sigma(f_vals):
    N = len(f_vals)
    if N == 0: return 0, 0, 0 # Protection against division-by-zero errors
    f = f_vals.mean()
    sigma = np.sqrt(f * (1 - f) / N)
    return f, sigma, N

# Split populations strictly by their quality tracks AND their BDT cut pass state
sig_pass = bdt_pass & (y_full == 1) #High Quality Tracks
bkg_pass = bdt_pass & (y_full == 0) #Low quality Tracks

f_sig_before, s_sig_before, N_sig_before = f_and_sigma(f_track_full[y_full == 1])
f_bkg_before, s_bkg_before, N_bkg_before = f_and_sigma(f_track_full[y_full == 0])
f_sig_after,  s_sig_after,  N_sig_after  = f_and_sigma(f_track_full[sig_pass])
f_bkg_after,  s_bkg_after,  N_bkg_after  = f_and_sigma(f_track_full[bkg_pass])

# 5. PRINT REPORT: Output a table
print(f"BDT selection threshold cut: {trkqual_cut:.2f}")
print()
print(f"{'Tracking Category':30s} {'N tracks':>8s} {'f_active':>10s} {'±σ':>8s}")
print("-" * 62)
print(f"{'Signal (before cut)':30s} {N_sig_before:8d} {f_sig_before:10.4f} {s_sig_before:8.4f}")
print(f"{'Signal (after cut)':30s} {N_sig_after:8d} {f_sig_after:10.4f} {s_sig_after:8.4f}")
print(f"{'Background (before cut)':30s} {N_bkg_before:8d} {f_bkg_before:10.4f} {s_bkg_before:8.4f}")
print(f"{'Background (after cut)':30s} {N_bkg_after:8d} {f_bkg_after:10.4f} {s_bkg_after:8.4f}")

BDT selection threshold cut: 0.92

Tracking Category              N tracks   f_active       ±σ
--------------------------------------------------------------
Signal (before cut)              303990     0.9463   0.0004
Signal (after cut)               172011     0.9592   0.0005
Background (before cut)          148696     0.9104   0.0007
Background (after cut)            31694     0.9458   0.0013
